**🔹 Step 1: Mount Drive**

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


**🔹 Step 2: Paths**

In [2]:
BASE_PATH = "/content/drive/MyDrive/ResearchModels/Paper1"
MODEL_PATH = f"{BASE_PATH}/models/distilbert_sst2"

**🔹 Step 3: Imports **

In [3]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import shap

from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast


**🔹 Step 4: Define prediction function for SHAP**

SHAP needs probabilities, not logits.

In [24]:
def shap_predict(texts):
    # Ensure texts is a list of strings for the tokenizer
    if isinstance(texts, np.ndarray):
        texts_to_tokenize = texts.flatten().tolist()
    else:
        texts_to_tokenize = list(texts) # Fallback, should ideally be list of strings already

    enc = tokenizer(
        texts_to_tokenize,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model(**enc)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)

    return probs.cpu().numpy()

**🔹 Step 5: Background data**

In [25]:
import numpy as np

background_texts = np.array(
    ["The movie was okay.", "The film was average."],
    dtype=object
)


**🔹 Step 6: Create SHAP explainer (TEXT EXPLAINER)**

In [32]:
import shap

explainer = shap.KernelExplainer(
    shap_predict,
    background_texts.reshape(-1, 1)
)


**🔹 Step 7: Run SHAP on sample sentences**

Use the same style of sentences as IG & Attention. **bold text**

In [35]:
texts = [
    "The movie was absolutely wonderful and engaging.",
    "The film was dull, boring, and poorly executed."
]

# Convert texts to a numpy array and reshape to (n_samples, 1)
texts_np = np.array(texts).reshape(-1, 1)

shap_values = explainer(texts_np)

  0%|          | 0/2 [00:00<?, ?it/s]

**🔹 Step 8: Visualize SHAP explanations**

In [36]:
shap.plots.text(shap_values[0])


**🔹 Step 9: Save SHAP output**

SHAP text plots are HTML-based. Save them.

In [40]:
save_path = "/content/drive/MyDrive/ResearchModels/Paper1/plots/shap_plots"
!mkdir -p $save_path

# Get the text plot HTML string directly
html_content = shap.plots.text(shap_values[0], display=False)

# Save the HTML content to a file
with open(f"{save_path}/shap_example.html", "w") as f:
    f.write(html_content)

print("SHAP output saved.")

SHAP output saved.
